# ⚡ Asenkron Scraping: Hız Testi

Bu notebook'ta web scraping'in en heyecanlı konularından birini işleyeceğiz: **Hız!**

İki yaklaşımı yarıştıracağız:
1. **Senkron (Synchronous):** `requests` kütüphanesi. Bir sayfa bitmeden diğerine geçmez.
2. **Asenkron (Asynchronous):** `aiohttp` kütüphanesi. Aynı anda (concurrent) çok sayıda istek atar.

**Senaryo:** `httpbin.org` sitesine 50 adet istek göndereceğiz ve geçen süreyi ölçeceğiz.

In [ ]:
# Gerekli kütüphaneler
import requests
import aiohttp
import asyncio
import time
import nest_asyncio

# Jupyter içinde asyncio döngüsü hatası almamak için
nest_asyncio.apply()

# Test URL'i (Gecikmeli yanıt veren bir endpoint kullanıyoruz)
# delay/1 -> Sayfa 1 saniye sonra yanıt verir
URL = "https://httpbin.org/delay/1"
TOTAL_REQUESTS = 10

## 🐢 Yöntem 1: Senkron (Requests)
Klasik `for` döngüsü ile istekleri sırayla atar.

In [ ]:
def sync_scrape():
    print("🐢 Senkron işlem başladı...")
    start_time = time.time()
    
    for i in range(TOTAL_REQUESTS):
        resp = requests.get(URL)
        # print(f"İstek {i+1} bitti") # Çıktıyı kirletmemek için kapalı
        
    end_time = time.time()
    duration = end_time - start_time
    print(f"✅ Senkron Bitti! Süre: {duration:.2f} saniye")
    return duration

# Testi çalıştır
sync_duration = sync_scrape()

### 🤔 Analiz
Her istek 1 saniye sürdüğü için ve 10 istek attığımız için toplam sürenin **yaklaşık 10 saniye** sürmesi normaldir. Çünkü hepsi sırasını bekledi.

## 🚀 Yöntem 2: Asenkron (aiohttp)
Python'ın `async/await` yapısını kullanarak istekleri paralele yakın bir şekilde atar.

In [ ]:
async def fetch(session, url):
    async with session.get(url) as response:
        return await response.text()

async def async_scrape():
    print("🚀 Asenkron işlem başladı...")
    start_time = time.time()
    
    async with aiohttp.ClientSession() as session:
        tasks = []
        for i in range(TOTAL_REQUESTS):
            task = asyncio.ensure_future(fetch(session, URL))
            tasks.append(task)
        
        # Tüm görevleri aynı anda başlat ve bitmelerini bekle
        await asyncio.gather(*tasks)
        
    end_time = time.time()
    duration = end_time - start_time
    print(f"✅ Asenkron Bitti! Süre: {duration:.2f} saniye")
    return duration

# Testi çalıştır
async_duration = asyncio.run(async_scrape())

### 😲 Analiz
Her istek yine 1 saniye sürdü. Ama 10 isteği neredeyse aynı anda attığımız için toplam süre **yaklaşık 1-1.5 saniye** sürdü!

**10 kat daha hızlı!** (İstek sayısı arttıkça bu fark daha da açılır).

## 📊 Sonuç Karşılaştırması

In [ ]:
import matplotlib.pyplot as plt

methods = ['Senkron (Requests)', 'Asenkron (aiohttp)']
times = [sync_duration, async_duration]

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, times, color=['red', 'green'])

plt.title(f'{TOTAL_REQUESTS} İstek İçin Performans Karşılaştırması')
plt.ylabel('Süre (Saniye)')

# Barların üzerine değerleri yaz
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}s',
             ha='center', va='bottom')

plt.show()